In [1]:
import pandas as pd
import numpy as np
from scipy import stats

file = 'Данные для тестового задания (1).xlsx'
aud = pd.read_excel(file, sheet_name='Данные об аудитории')
aud['date'] = pd.to_datetime(aud['date'])

In [4]:
#1
mau = aud['user_id'].nunique()
print('MAU:', mau)

MAU: 7639


In [5]:
#2
daily_users = aud.groupby('date')['user_id'].nunique()
dau = daily_users.mean()
print('DAU:', round(dau, 2))

DAU: 560.47


In [6]:
#3
cohort = set(aud.loc[aud['date'].dt.date == pd.Timestamp('2023-11-01').date(), 'user_id'])
day1 = set(aud.loc[aud['date'].dt.date == pd.Timestamp('2023-11-02').date(), 'user_id'])
retention_day1 = len(cohort & day1) / len(cohort)
print('Day-1 retention:', round(retention_day1*100, 2), '%')

Day-1 retention: 26.65 %


#4
Продукт 1 значительно лучше удерживает пользователей после 2-го дня. У продукта 2 retention быстро снижается и достигает 0% примерно к 5-му дню.

In [7]:
#5
views_by_user = aud.groupby('user_id')['view_adverts'].sum()
conversion = (views_by_user > 0).mean()
print('Conversion:', round(conversion*100, 2), '%')

Conversion: 46.31 %


In [8]:
#6
avg_views = aud['view_adverts'].sum() / mau
print('Average views per user:', round(avg_views, 2))

Average views per user: 2.87


In [9]:
#7
promoters = 1200
detractors = 500
total = 2000

nps = (promoters / total - detractors / total) * 100
print(f"NPS = {nps:.0f}%")

NPS = 35%


In [10]:
ab = pd.read_excel(file, sheet_name='Данные АБ тестов')

for exp, g in ab.groupby('experiment_num'):
    control = g.loc[g['experiment_group'] == 'control', 'revenue']
    test = g.loc[g['experiment_group'] == 'test', 'revenue']
    result = stats.ttest_ind(control, test, equal_var=False)
    print(f'Experiment {exp}:')
    print('  control ARPU =', round(control.mean(), 2))
    print('  test ARPU    =', round(test.mean(), 2))
    print('  p-value      =', round(result.pvalue, 4))

Experiment 1:
  control ARPU = 722.46
  test ARPU    = 665.74
  p-value      = 0.689
Experiment 2:
  control ARPU = 704.65
  test ARPU    = 332.93
  p-value      = 0.0011
Experiment 3:
  control ARPU = 663.21
  test ARPU    = 998.67
  p-value      = 0.0603


In [11]:
#9-10
lst = pd.read_excel(file, sheet_name='Листеры')
arpu = lst['revenue'].sum() / lst['user_id'].nunique()
median_age = lst.drop_duplicates('user_id')['age'].median()
print('ARPU:', round(arpu, 2))
print('Median age:', median_age)

ARPU: 156.48
Median age: 28.0


In [13]:
#18
n_a, x_a = 100_047_501, 1003
n_b, x_b = 100_001_055, 1099
p_a, p_b = x_a/n_a, x_b/n_b
p_pool = (x_a+x_b)/(n_a+n_b)
se = np.sqrt(p_pool*(1-p_pool)*(1/n_a + 1/n_b))
z = (p_b-p_a)/se
p_value = 2*(1-stats.norm.cdf(abs(z)))
relative_lift = (p_b-p_a)/p_a

print('Conversion A:', p_a)
print('Conversion B:', p_b)
print('Relative lift B vs A:', relative_lift)
print('z:', z)
print('p-value:', p_value)

Conversion A: 1.0025237911739544e-05
Conversion B: 1.0989884056723202e-05
Relative lift B vs A: 0.09622177084237159
z: 2.1045506019397746
p-value: 0.03533044544854258
